# Self-Consistency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/02-reasoning/11_self_consistency.ipynb)

**Category:** Reasoning & Logic  
**Technique #11**

---

## 📋 Description

Self-Consistency is an ensemble technique that generates multiple reasoning paths for the same problem and selects the most frequent answer. Instead of relying on a single greedy decode, it samples diverse reasoning paths and uses majority voting to determine the final answer.

**When to use:**
- When accuracy is more important than speed
- Complex reasoning problems with multiple valid paths
- When you can afford multiple API calls
- Math word problems and logical reasoning

## 🔧 How It Works

```
Self-Consistency Process:
┌─────────────────────────────────────────────────────┐
│                    Problem                          │
│         "What is 15 + 27 - 8?"                    │
└──────────────────┬──────────────────────────────────┘
                   │
         ┌─────────┼─────────┐
         ▼         ▼         ▼
    ┌────────┐ ┌────────┐ ┌────────┐
    │Path 1  │ │Path 2  │ │Path 3  │
    │15+27=42│ │15+27=42│ │15+27=42│
    │42-8=34 │ │42-8=34 │ │42-8=34 │
    │Ans: 34 │ │Ans: 34 │ │Ans: 34 │
    └────────┘ └────────┘ └────────┘
         │         │         │
         └─────────┼─────────┘
                   ▼
         ┌───────────────────┐
         │  Majority Voting  │
         │    Answer: 34     │
         └───────────────────┘
```

**Key Steps:**
1. Generate multiple outputs with temperature > 0
2. Extract answers from each reasoning path
3. Count frequency of each answer
4. Select the most common answer

In [ ]:
import osfrom getpass import getpass# Install required packages (uncomment if needed)# !pip install openai -q# Set up OpenAI API key securelyif "OPENAI_API_KEY" not in os.environ:    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")# Import OpenAIfrom openai import OpenAIclient = OpenAI()def get_completion(prompt, model="gpt-4", temperature=0.7):    """Helper function to get completions from OpenAI API"""    try:        response = client.chat.completions.create(            model=model,            messages=[                {"role": "system", "content": "You are a helpful assistant."},                {"role": "user", "content": prompt}            ],            temperature=temperature        )        return response.choices[0].message.content    except Exception as e:        return f"Error: {str(e)}"print("✅ Setup complete! Ready to experiment with prompts.")

## 💡 Basic Example

Demonstrating Self-Consistency on a math word problem by generating multiple reasoning paths.

**Problem:** A library has 240 books. They receive 85 more and then lend out 120. How many books remain?

In [ ]:
import refrom collections import Counterdef extract_answer(text):    """Extract numerical answer from reasoning text"""    # Look for patterns like "The answer is X" or "Answer: X"    patterns = [        r'[Tt]he answer is[:\s]+(\d+)',        r'[Aa]nswer[:\s]+(\d+)',        r'=\s*(\d+)\s*$',  # Ending with = number        r'(?:total|remain|left|have)[^\d]*(\d+)[^\d]*$',  # Keywords before number at end    ]    for pattern in patterns:        match = re.search(pattern, text)        if match:            return int(match.group(1))    # Fallback: find last number in text    numbers = re.findall(r'\d+', text)    if numbers:        return int(numbers[-1])    return None# Problem setupproblem = """A library has 240 books. They receive 85 more and then lend out 120. How many books remain?Let's think step by step."""print("="*60)print("🔄 Self-Consistency Demo")print("="*60)print(f"Problem: {problem.strip()}")print()# Generate multiple reasoning pathsnum_paths = 5temperature = 0.7paths = []print(f"Generating {num_paths} reasoning paths (temperature={temperature}):")for i in range(num_paths):    response = get_completion(problem, temperature=temperature)    answer = extract_answer(response)    paths.append((response, answer))    print(f"--- Path {i+1} ---")    print(response[:200] + "..." if len(response) > 200 else response)    print(f"Extracted Answer: {answer}")    print()# Majority votinganswers = [p[1] for p in paths if p[1] is not None]if answers:    answer_counts = Counter(answers)    most_common = answer_counts.most_common(1)[0]    print("="*60)    print("📊 Voting Results:")    print("="*60)    for ans, count in answer_counts.most_common():        print(f"  Answer {ans}: {count} votes")    print(f"✅ Final Answer (majority): {most_common[0]}")    print(f"   Confidence: {most_common[1]}/{num_paths} = {most_common[1]/num_paths*100:.1f}%")

## 🌍 Real-World Example

**Scenario:** Medical diagnosis support - using self-consistency to improve diagnostic accuracy.

Multiple reasoning paths help catch different diagnostic considerations.

In [ ]:
# Real-World: Medical Diagnosis with Self-Consistencydiagnosis_problem = """Patient: 58-year-old femaleSymptoms: Fatigue, increased thirst, frequent urination, blurred vision, slow-healing cutsFamily history: Type 2 diabetes in motherBMI: 31What is the most likely diagnosis? Provide reasoning.Let's think through this systematically."""def extract_diagnosis(text):    """Extract diagnosis from medical reasoning"""    text_lower = text.lower()    if 'diabetes' in text_lower or 'type 2' in text_lower:        return 'Type 2 Diabetes'    elif 'prediabetes' in text_lower:        return 'Prediabetes'    elif 'kidney' in text_lower:        return 'Kidney Disease'    elif 'thyroid' in text_lower:        return 'Thyroid Disorder'    else:        return 'Other'print("="*60)print("🏥 Medical Diagnosis with Self-Consistency")print("="*60)# Generate multiple diagnostic pathsnum_paths = 5paths = []for i in range(num_paths):    response = get_completion(diagnosis_problem, temperature=0.7)    diagnosis = extract_diagnosis(response)    paths.append((response, diagnosis))    print(f"--- Diagnostic Path {i+1} ---")    print(response[:300] + "..." if len(response) > 300 else response)    print(f"Diagnosis: {diagnosis}")# Aggregate resultsdiagnoses = [p[1] for p in paths]diagnosis_counts = Counter(diagnoses)print("" + "="*60)print("📊 Diagnostic Consensus:")print("="*60)for diag, count in diagnosis_counts.most_common():    bar = "█" * count    print(f"  {diag:20} | {bar} ({count}/{num_paths})")most_common = diagnosis_counts.most_common(1)[0]print(f"✅ Most Likely Diagnosis: {most_common[0]}")print(f"   Confidence: {most_common[1]/num_paths*100:.1f}%")

## ⚠️ Failure Case

Self-Consistency can fail when:
1. All paths converge on the same wrong answer
2. Answer extraction is unreliable
3. The problem has genuinely ambiguous answers
4. Temperature is too low (no diversity) or too high (nonsensical)
5. Cost/latency constraints limit the number of samples

In [ ]:
# Failure Case: When Self-Consistency fails# Problem where model might consistently get wrongambiguous_problem = """If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?Let's think step by step."""print("="*60)print("⚠️ Self-Consistency Failure Case")print("="*60)print(f"Problem: {ambiguous_problem.strip()}")print("This is a classic 'bat and ball' style problem that tricks intuition.")print("The correct answer is 5 minutes (each machine makes 1 widget in 5 minutes).")print("Let's see if self-consistency helps:")# Generate pathspaths = []for i in range(5):    response = get_completion(ambiguous_problem, temperature=0.5)    # Extract numeric answer    numbers = re.findall(r'\d+', response)    answer = int(numbers[-1]) if numbers else None    paths.append((response, answer))    print(f"Path {i+1} Answer: {answer}")    print(f"Reasoning: {response[:150]}...")    print()answers = [p[1] for p in paths if p[1] is not None]if answers:    answer_counts = Counter(answers)    print("Vote distribution:", dict(answer_counts))    print("💡 Note: If most paths say 100, self-consistency won't help!")

## 📊 Benchmark

| Dataset | Greedy Decode | Self-Consistency (5 paths) | Self-Consistency (10 paths) |
|---------|---------------|---------------------------|----------------------------|
| GSM8K | 56.5% | 68.7% | 72.2% |
| SVAMP | 68.9% | 76.4% | 79.8% |
| AQuA | 35.8% | 42.1% | 45.3% |
| StrategyQA | 73.9% | 79.2% | 81.5% |

**Key Findings:**
- Consistent gains across all reasoning datasets
- Diminishing returns after ~10 samples
- Best temperature: 0.5-0.7 for diverse but coherent paths
- Trade-off: Accuracy vs. Cost (N× API calls)

## 🎮 Interactive Playground

Experiment with your own prompts below!

In [ ]:
# 🎮 Interactive Playground# Modify the prompt below and run to see resultsyour_prompt = """# Your prompt here"""# Get responseresponse = get_completion(your_prompt)print("="*50)print("📝 Response:")print("="*50)print(response)

## 💡 Tips & Tricks

**Implementation Tips:**
- ✓ Use temperature 0.5-0.7 for diverse reasoning
- ✓ Generate 5-10 paths for good balance
- ✓ Implement robust answer extraction
- ✓ Consider confidence thresholds (e.g., require >60% agreement)
- ✗ Don't use temperature 0 (no diversity)
- ✗ Don't sample too many paths (>20 has diminishing returns)

**Cost Considerations:**
- Self-consistency costs N× more than single generation
- Consider using smaller models for initial screening
- Cache results for repeated queries

## 📚 References

1. **Self-Consistency Improves Chain of Thought Reasoning in Language Models** (Wang et al., 2022)
   - [Paper](https://arxiv.org/abs/2203.11171)

2. **DiVeRSe: Making Language Models Better Reasoners** (Li et al., 2022)
   - [Paper](https://arxiv.org/abs/2210.09297)

3. **Learn Prompting: Self-Consistency**
   - [Tutorial](https://learnprompting.org/docs/intermediate/self_consistency)